# Footage Engine — YouTube ingest

Clones the repo, reconstructs `.env` from Kaggle secrets, pulls YouTube cookies, and ingests one or more videos into scene chunks.

## Secrets to attach (Add-ons → Secrets)

Required:

| Secret | Purpose |
|---|---|
| `DATABASE_URL` | Shared job/media DB. **Must not be SQLite** — a Kaggle session's local file vanishes when it ends. |
| `VECTOR_STORE` | `zilliz`. A remote worker with `in_memory` finds nothing. |
| `ZILLIZ_URI`, `ZILLIZ_TOKEN`, `ZILLIZ_COLLECTION_NAME` | X-CLIP collection (512d). |
| `INGEST_URLS` | Comma-separated YouTube URLs. |
| `YT_COOKIES_URL` | HTTPS endpoint returning a Netscape `cookies.txt`. Never hardcode this — see the security note below. |

Required only when `EMBEDDING_BACKEND=qwen`:

| Secret | Purpose |
|---|---|
| `QWEN_ZILLIZ_URI`, `QWEN_ZILLIZ_TOKEN`, `QWEN_ZILLIZ_COLLECTION_NAME` | Qwen collection (2048d — cannot share the X-CLIP collection). |

Required only when `STORAGE_BACKEND=gdrive`:

| Secret | Purpose |
|---|---|
| `GDRIVE_OAUTH_CREDENTIALS_JSON` | Whole `authorized_user` token JSON, minified onto one line. |
| `GDRIVE_FOLDER_ID` | Destination folder. |
| `UPLOAD_RAW_TO_STORAGE`, `UPLOAD_CHUNKS_TO_STORAGE` | `true` to upload. |

## Security

This kernel is set `is_private: true`. It still clones a public GitHub repo and writes secrets into a working tree on Kaggle's filesystem, so:

- Never put a cookie URL, token, or key in a committed file. Everything comes from `kaggle_secrets`.
- `kernel-metadata.json` and this notebook are safe to commit; the values are not.
- Kaggle secrets are readable by **anyone who can open the kernel**. Keep it private and grant access only to collaborators.

In [ ]:
# !pip install -q --upgrade pip
# Kaggle ships torch + cv2 already; only add what is genuinely missing so
# the install does not pull a second multi-GB torch wheel over the pinned one.
import importlib, subprocess, sys

NEEDED = [
    ("transformers", "transformers>=4.46,<4.58"),
    ("sentence_transformers", "sentence-transformers"),
    ("scenedetect", "scenedetect[opencv]"),
    ("pymilvus", "pymilvus"),
    ("sqlalchemy", "sqlalchemy"),
    ("pydantic_settings", "pydantic-settings"),
    ("pydantic", "pydantic>=2.5"),
    ("PIL", "pillow"),
    ("cv2", "opencv-python-headless"),
    ("imagekitio", "imagekitio"),
    ("yt_dlp", "yt-dlp"),
    # YouTube serves media only to clients that solve its JS challenge, which
    # needs BOTH the EJS components and a JS runtime. yt-dlp enables only deno
    # by default and refuses node.
    ("yt_dlp_ejs", "yt-dlp-ejs"),
    ("psycopg2", "psycopg2-binary"),
    ("dotenv", "python-dotenv"),
    ("requests", "requests"),
    ("googleapiclient", "google-api-python-client"),
    ("google.oauth2", "google-auth"),
]

missing = [spec for module, spec in NEEDED if importlib.util.find_spec(module) is None]
if missing:
    print(f"installing {len(missing)}: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import os
import stat

# deno: not preinstalled on Kaggle, and youtube.py only probes ~/.deno/bin/deno.
DENO = os.path.expanduser("~/.deno/bin/deno")
if not os.path.exists(DENO):
    print("installing deno...")
    subprocess.run(
        "curl -fsSL https://deno.land/install.sh | sh -s -- -y",
        shell=True, check=True,
        executable="/bin/bash",
    )
os.chmod(DENO, os.stat(DENO).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print("deno:", subprocess.run([DENO, "--version"], capture_output=True, text=True).stdout.split()[1])

# deno must be importable by the yt-dlp subprocess as well as by path.
os.environ["PATH"] = os.path.dirname(DENO) + ":" + os.environ.get("PATH", "")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("yt-dlp-ejs:", __import__("importlib.metadata", fromlist=["version"]).version("yt-dlp-ejs"))

In [ ]:
# Build .env from, in order of preference:
#   1. the private `footage-engine-secrets` Kaggle dataset (CLI-manageable)
#   2. Kaggle UI secrets (masked in logs, but must be added by hand)
# The dataset wins because `kaggle kernels push` cannot attach UI secrets at all.
import glob
import os
import shutil

DATASET_ENV = next(iter(glob.glob("/kaggle/input/*/runtime.env")), None)

if not DATASET_ENV:
    # Fallback: dataset_sources did not mount (Kaggle does not always attach a
    # private dataset to a private kernel). The in-kernel CLI is already
    # authenticated as the notebook owner, so fetching it directly works.
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        target = "annasblackhat/footage-engine-secrets"
        api.dataset_download_files(target, path=".", unzip=True, quiet=True)
        downloaded = next(iter(glob.glob("**/runtime.env", recursive=True)), None)
        if downloaded and downloaded != "runtime.env":
            shutil.move(downloaded, "runtime.env")
        if os.path.isfile("runtime.env"):
            DATASET_ENV = os.path.abspath("runtime.env")
            print(f"dataset not mounted; fetched it via the Kaggle API -> {DATASET_ENV}")
    except Exception as exc:
        print(f"dataset fetch fallback failed: {type(exc).__name__}: {exc}")

HAVE_SECRETS = True
try:
    from kaggle_secrets import UserSecretClient
except Exception:
    HAVE_SECRETS = False

if DATASET_ENV:
    shutil.copy(DATASET_ENV, ".env")
    print(f"loaded .env from private dataset: {DATASET_ENV}")
    print("SECURITY NOTE: dataset values are NOT masked in logs, unlike UI secrets.")
elif HAVE_SECRETS:
    print("no dataset attached; falling back to Kaggle UI secrets")
else:
    raise SystemExit(
        "No credential source. Expected the private dataset at "
        "/kaggle/input/*/runtime.env, or UI secrets. See kaggle/README.md."
    )


def secret(name):
    """First non-empty value wins: dataset file, then UI secret, then None."""
    if DATASET_ENV:
        with open(DATASET_ENV, encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, _, value = line.partition("=")
                    if key.strip() == name and value.strip():
                        return value.strip()
    if HAVE_SECRETS:
        try:
            return UserSecretClient().get_secret(name)
        except Exception:
            return None
    return None


# name -> default written when the key is absent everywhere, so an omitted
# optional key degrades to a documented default instead of an empty value.
ENV_DEFAULTS = {
    "VECTOR_STORE": "zilliz",
    "EMBEDDING_BACKEND": "xclip",
    "EMBEDDING_DIMENSION": "512",
    "DEFAULT_EMBEDDING_MODEL": "microsoft/xclip-base-patch32",
    "EMBEDDING_DEVICE": "auto",
    "NUM_WORKERS": "2",
    "STORAGE_BACKEND": "local",
    "UPLOAD_RAW_TO_STORAGE": "false",
    "UPLOAD_CHUNKS_TO_STORAGE": "false",
    "GDRIVE_SCOPES": "https://www.googleapis.com/auth/drive",
    "GDRIVE_URL_TEMPLATE": "https://lh3.googleusercontent.com/d/{file_id}",
    "CHUNK_THRESHOLD_SEC": "45.0",
}

HARD_REQUIRED = [
    "DATABASE_URL",
    "VECTOR_STORE",
    "ZILLIZ_URI",
    "ZILLIZ_TOKEN",
    "ZILLIZ_COLLECTION_NAME",
    "INGEST_URLS",
    "YT_COOKIES_URL",
]

# Anything else the notebook advertises (Qwen collection, GDRIVE_*, optional LLM
# keys) is copied through when present, and skipped when not.
OPTIONAL = [
    "QWEN_ZILLIZ_URI", "QWEN_ZILLIZ_TOKEN", "QWEN_ZILLIZ_COLLECTION_NAME",
    "STORAGE_BACKEND", "GDRIVE_OAUTH_CREDENTIALS_JSON", "GDRIVE_FOLDER_ID",
    "GDRIVE_DRIVE_ID", "GDRIVE_SCOPES", "GDRIVE_URL_TEMPLATE",
    "UPLOAD_RAW_TO_STORAGE", "UPLOAD_CHUNKS_TO_STORAGE",
    "LOCAL_STORAGE_DIR", "IMAGEKIT_PUBLIC_KEY", "IMAGEKIT_PRIVATE_KEY",
    "IMAGEKIT_URL_ENDPOINT",
    "GEMINI_API_KEY", "LLM_API_KEY", "OPENAI_API_KEY", "LLM_BASE_URL", "LLM_MODEL",
    "INGEST_ENTITY", "INGEST_ENTITY_TYPE",
]

# Export everything into the process env so subprocesses (the ingest script)
# inherit it, not just this cell.
resolved, missing = {}, []
for name in HARD_REQUIRED:
    value = secret(name)
    if not value:
        missing.append(name)
    else:
        resolved[name] = value
for name, default in ENV_DEFAULTS.items():
    resolved.setdefault(name, secret(name) or default)
for name in OPTIONAL:
    value = secret(name)
    if value:
        resolved[name] = value

if missing:
    raise SystemExit(
        "Missing required keys: " + ", ".join(missing) + "\n"
        "Rebuild the private dataset and re-run:\n"
        "  python scripts/build_kaggle_dataset.py --set <KEY>=<VALUE> ...\n"
        "  kaggle datasets version -p kaggle/dataset -m refresh\n"
        "or attach them under Add-ons -> Secrets."
    )

os.makedirs("Footage-Engine", exist_ok=True)
if not DATASET_ENV:  # dataset supplied .env; only synthesise when building from secrets
    with open("Footage-Engine/.env", "w") as fh:
        fh.write("\n".join(f"{k}={resolved[k]}" for k in sorted(resolved)) + "\n")

os.environ.update(resolved)

# Values are secret; key names are not, so only the names are logged.
print("resolved keys:", ", ".join(sorted(resolved)))

In [ ]:
REPO = "https://github.com/ANNASBlackHat/Footage-Engine.git"

subprocess.run(["git", "clone", "--depth", "1", REPO, "Footage-Engine"], check=True)
os.chdir("Footage-Engine")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[video,gdrive,mcp]"], check=True)

import footage_engine

from footage_engine.config import get_settings

cfg = get_settings()
print("db        :", cfg.DATABASE_URL.split("@")[-1])
print("vector    :", cfg.VECTOR_STORE, "| backend:", cfg.EMBEDDING_BACKEND)
print("storage   :", cfg.STORAGE_BACKEND, "| raw:", cfg.UPLOAD_RAW_TO_STORAGE, "| chunks:", cfg.UPLOAD_CHUNKS_TO_STORAGE)
print("embed dim :", cfg.EMBEDDING_DIMENSION)

if cfg.STORAGE_BACKEND == "gdrive":
    from footage_engine.storage import get_storage_backend

    backend = get_storage_backend(cfg)
    who = backend.service.about().get(fields="user,storageQuota").execute()
    quota = who.get("storageQuota", {})
    print("drive as  :", who["user"]["emailAddress"])
    print("identity  :", backend.identity)
    print("quota     : limit=%s usage=%s" % (quota.get("limit"), quota.get("usage")))
    if backend.identity == "service_account":
        print("WARNING: a service account has storageQuota.limit == 0; Drive will reject uploads.")
else:
    print("WARNING: STORAGE_BACKEND is not gdrive; nothing will be uploaded to Drive.")

In [ ]:
# Pull the Netscape cookies.txt before any yt-dlp call. Kaggle has no browser
# profile, so YOUTUBE_COOKIES_FROM_BROWSER cannot work here.
import shutil

cookies_url = os.environ["YT_COOKIES_URL"]
COOKIE_PATH = "yt_cookie.txt"
subprocess.run(
    ["curl", "-sL", "--fail", "--max-time", "60", cookies_url, "-o", COOKIE_PATH],
    check=True,
)

# A captive portal or expired worker key returns 200 with an HTML body. Catch
# that here rather than letting yt-dlp fail with a baffling cookie error.
head = open(COOKIE_PATH, encoding="utf-8", errors="replace").read(400).lstrip()
if not head.startswith("# Netscape HTTP Cookie File") and ".youtube.com" not in head:
    print("---- first 200 bytes of the response ----")
    print(open(COOKIE_PATH, encoding="utf-8", errors="replace").read(200))
    raise SystemExit(
        "Cookie fetch did not return a cookies.txt. Check that YT_COOKIES_URL is "
        "current — if the worker's key was rotated, this is what a stale one looks like."
    )

cookie_bytes = os.path.getsize(COOKIE_PATH)
domains = {ln.split("\t")[0] for ln in open(COOKIE_PATH, encoding="utf-8", errors="replace") if ln.strip() and not ln.startswith("#")}
print(f"cookies.txt: {cookie_bytes} bytes, domains={sorted(d for d in domains if d)[:8]}")

# Config reads the path; the file itself is gitignored-equivalent (untracked,
# and this is an ephemeral Kaggle filesystem).
with open(".env", "a") as fh:
    fh.write(f"YOUTUBE_COOKIES={os.path.abspath(COOKIE_PATH)}\n")
os.environ["YOUTUBE_COOKIES"] = os.path.abspath(COOKIE_PATH)

In [ ]:
urls = [u.strip() for u in os.environ["INGEST_URLS"].split(",") if u.strip()]
entity = os.environ.get("INGEST_ENTITY") or None
entity_type = os.environ.get("INGEST_ENTITY_TYPE", "other")

print(f"ingesting {len(urls)} video(s)")
for i, url in enumerate(urls, 1):
    print(f"\n{'=' * 70}\n[{i}/{len(urls)}] {url}\n{'=' * 70}")
    argv = [sys.executable, "scripts/ingest_youtube_video.py", url]
    if entity:
        argv += ["--entity", entity, "--entity-type", entity_type]
    result = subprocess.run(argv)
    if result.returncode != 0:
        # Keep going: one bad URL (removed video, region block) should not throw
        # away the chunks already produced by the videos before it.
        print(f"!! ingest failed rc={result.returncode} for {url} — continuing")

# Qwen lives in its own 2048d collection, so a second embedding for the same
# media is a backfill, not a re-ingest: media_items has a unique constraint on
# source_url, so re-running the ingest is a no-op once the item exists.
print("\n[backfill] adding Qwen embeddings where they are missing...")
# --chunk-batch bounds peak RAM: Qwen decodes every frame of every chunk in a
# call before encoding any, so a whole long item in one pass is ~1760 1080p frames.
_backfill = subprocess.run([sys.executable, "scripts/backfill_qwen.py", "--chunk-batch", "12"])
if _backfill.returncode != 0:
    print(f"!! qwen backfill rc={_backfill.returncode} — X-CLIP data is unaffected")

print("\nall done")

In [ ]:
# Post-run sanity check: how much actually landed, so a silent failure is visible
# in the notebook rather than only in the DB.
from sqlalchemy import text
from footage_engine.models.db import get_engine

# get_engine, not create_engine: the configured URL uses the postgres://
# scheme, which plain create_engine cannot resolve.
engine = get_engine(cfg.DATABASE_URL)
with engine.connect() as conn:
    for table in ("media_items", "chunks", "jobs"):
        try:
            n = conn.execute(text(f"SELECT count(*) FROM {table}")).scalar()
            print(f"{table:14} {n}")
        except Exception as exc:
            print(f"{table:14} n/a ({type(exc).__name__})")

    try:
        rows = conn.execute(
            text("SELECT provider, status, storage_path FROM media_items ORDER BY ingested_at DESC LIMIT 5")
        ).fetchall()
        for r in rows:
            print(dict(r._mapping))
    except Exception:
        pass

print("\nNote: SQLite would show 0 here even after a successful run — its file does")
print("not survive the session. Use a shared Postgres DATABASE_URL to verify remotely.")
# Kaggle saves the whole working directory as kernel output, and the credential
# files live in it. Scrub by absolute path from the session root: cwd is
# Footage-Engine/ by now, so a relative "runtime.env" would miss the copy that
# the dataset fallback wrote one level up.
import os as _os

ROOT = "/kaggle/working" if _os.path.isdir("/kaggle/working") else _os.getcwd()
TARGETS = {"runtime.env", ".env", "yt_cookie.txt"}
removed = 0
for dirpath, _dirnames, filenames in _os.walk(ROOT):
    for name in filenames:
        if name in TARGETS:
            full = _os.path.join(dirpath, name)
            try:
                _os.remove(full)
                removed += 1
                print(f"scrubbed {full}")
            except OSError as exc:
                print(f"could not scrub {full}: {exc}")
print(f"scrub removed {removed} credential file(s) from kernel output")
